# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# 1. Ranked Actions + Reason Codes

## Objective

The purpose of this playbook is to convert model predictions into clear and actionable recommendations.

Instead of only predicting an action label, the model provides a ranked list of content items together with a reason code that explains why each recommendation was made.

The recommendations are intended to support SEO teams during content planning. Final publication decisions should always include human review.

In [1]:
import pandas as pd

df = pd.read_csv("baseline_action_score.csv")

print(df.shape)

df.head()

(30000, 49)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,ctr_bucket,update_bucket,score,reason_code,action
0,content_5feee3994adb,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,transactional,3590.0,22780.0,...,0.0,good,page_3_5,down,-89.1,Very Low,181-365 Days,100,STALE_LOW_CTR,Refresh Content
1,content_1816f5ff12ac,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,5690.0,37511.0,...,0.0,good,striking,stable,14.9,Very Low,91-180 Days,90,STALE_LOW_CTR,Refresh Content
2,content_9648b7053d6f,client_19581e27de,30.0,0.07,LOW,0.06,keyword article,informational,NaN,NaN,...,0.0,good,page_1,stable,-10.5,Very Low,91-180 Days,90,STALE_LOW_CTR,Refresh Content
3,content_0c714e1e3130,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,5988.0,38009.0,...,0.0,good,page_3_5,down,-24.2,Very Low,91-180 Days,90,STALE_LOW_CTR,Refresh Content
4,content_19844deccf29,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,6612.0,45142.0,...,0.0,good,page_3_5,down,-38.6,Very Low,91-180 Days,90,STALE_LOW_CTR,Refresh Content


In [2]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'ctr_bucket', 'update_bucket', 'score', 'reason_code', 'action']


In [3]:
queue = df.copy()

queue = queue.sort_values(
    by="score",
    ascending=False
)

queue = queue.reset_index(drop=True)

queue.head(20)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,ctr_bucket,update_bucket,score,reason_code,action
0,content_5feee3994adb,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,transactional,3590.0,22780.0,...,0.00,good,page_3_5,down,-89.1,Very Low,181-365 Days,100,STALE_LOW_CTR,Refresh Content
1,content_70450b1c27ae,client_3fdba35f04,40.0,0.09,LOW,0.02,keyword article,commercial,1287.0,8389.0,...,0.00,good,page_3_5,down,-85.2,Very Low,91-180 Days,90,STALE_LOW_CTR,Refresh Content
2,content_5096a9d25fe5,client_6208ef0f77,10.0,0.14,LOW,0.00,keyword article,informational,7180.0,47955.0,...,0.00,excellent,page_3_5,down,-56.9,Very Low,91-180 Days,90,STALE_LOW_CTR,Refresh Content
3,content_d20b4e742dc6,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,3684.0,26361.0,...,0.00,good,page_1,down,-39.7,Very Low,91-180 Days,90,STALE_LOW_CTR,Refresh Content
4,content_c7fbadd2bf54,client_19581e27de,70.0,0.03,LOW,0.10,keyword article,transactional,NaN,NaN,...,0.00,good,page_1,down,-25.8,Very Low,91-180 Days,90,STALE_LOW_CTR,Refresh Content
5,content_5a67177a8e65,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,5424.0,35547.0,...,0.00,good,page_3_5,stable,4.4,Very Low,91-180 Days,90,STALE_LOW_CTR,Refresh Content
6,content_37c34ce699c3,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.00,good,page_1,down,-41.5,Very Low,91-180 Days,90,STALE_LOW_CTR,Refresh Content
7,content_88d367c507a3,client_3fdba35f04,10.0,0.24,LOW,0.00,keyword article,informational,1476.0,9302.0,...,0.00,excellent,page_3_5,stable,-13.7,Very Low,91-180 Days,90,STALE_LOW_CTR,Refresh Content
8,content_dec180daa24f,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,6929.0,44119.0,...,0.00,good,page_3_5,up,84.2,Very Low,91-180 Days,90,STALE_LOW_CTR,Refresh Content
9,content_4f81a5e449e2,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,5057.0,32900.0,...,0.00,good,page_3_5,stable,-16.6,Very Low,91-180 Days,90,STALE_LOW_CTR,Refresh Content


In [4]:
def reason_code(row):

    if row["score"] >= 90:
        return "High predicted value"

    elif row["score"] >= 75:
        return "Strong improvement opportunity"

    elif row["score"] >= 60:
        return "Moderate optimization recommended"

    else:
        return "Low priority"

queue["reason_code"] = queue.apply(reason_code, axis=1)

queue[
    [
        "content_id",
        "client_id",
        "score",
        "reason_code"
    ]
].head(20)

,content_id,client_id,score,reason_code
0,content_5feee3994adb,client_7f2253d7e2,100,High predicted value
1,content_70450b1c27ae,client_3fdba35f04,90,High predicted value
2,content_5096a9d25fe5,client_6208ef0f77,90,High predicted value
3,content_d20b4e742dc6,client_19581e27de,90,High predicted value
4,content_c7fbadd2bf54,client_19581e27de,90,High predicted value
5,content_5a67177a8e65,client_6208ef0f77,90,High predicted value
6,content_37c34ce699c3,client_19581e27de,90,High predicted value
7,content_88d367c507a3,client_3fdba35f04,90,High predicted value
8,content_dec180daa24f,client_6208ef0f77,90,High predicted value
9,content_4f81a5e449e2,client_6208ef0f77,90,High predicted value


### Observation

Content with higher scores is ranked first because these pages are expected to provide the greatest improvement opportunity.

The reason code explains why each page appears in the queue and helps reviewers understand the recommendation.

# 2. Intended Use and Limits

## Intended Use

This playbook is designed to support SEO analysts and content managers when prioritizing pages for improvement.

The recommendations should be used to:

- prioritize content updates
- identify high-value pages
- support content planning
- allocate editorial resources

---

## Limits

The model should not be used as the only decision-making tool.

It does not consider:

- business priorities
- legal requirements
- brand strategy
- current marketing campaigns
- manually collected expert knowledge

Human judgement is required before implementing recommendations.

In [5]:
print(queue["reason_code"].value_counts())

reason_code
Low priority                         19740
Moderate optimization recommended     6590
Strong improvement opportunity        2977
High predicted value                   693
Name: count, dtype: int64


# 3. Human Review + No-Go List

## Human Review Checklist

Before acting on any recommendation, reviewers should verify:

- The content is still published.
- The page belongs to the correct client.
- The recommendation matches business goals.
- Important information will not be removed.
- SEO changes comply with company guidelines.

---

## No-Go Cases

The recommendations should NOT be applied automatically when:

- legal or compliance approval is required
- medical or financial content is involved
- sensitive information is present
- business strategy has recently changed
- the page has already been updated recently

These situations require manual review.

In [6]:
print("Unique Clients:", queue["client_id"].nunique())

print("Unique Pages:", queue["content_id"].nunique())

Unique Clients: 32
Unique Pages: 30000


# 4. Monitoring and Retraining

The model should be monitored regularly.

Retraining should be considered when:

- prediction accuracy decreases
- new content types appear
- user behaviour changes
- SEO strategy changes
- search engine algorithms change
- data drift is detected
- large amounts of new training data become available

Regular monitoring helps maintain reliable recommendations.

In [7]:
print(queue.describe())

       search_volume   competition           cpc    word_count     char_count  \
count   27532.000000  27532.000000  27532.000000  22301.000000   22301.000000   
mean      158.882391      0.146954      0.485342   3107.760325   20665.277835   
std      1518.270825      0.285241      2.101560   1452.382598   10115.344042   
min         0.000000      0.000000      0.000000      8.000000      40.000000   
25%         0.000000      0.000000      0.000000   2413.000000   15644.000000   
50%        10.000000      0.000000      0.000000   2877.000000   19116.000000   
75%        20.000000      0.130000      0.000000   3666.000000   24011.000000   
max     74000.000000      1.000000    100.360000   9546.000000  111158.000000   

       impressions_90d    clicks_90d  pageviews_90d  sessions_90d  \
count     30000.000000  30000.000000   30000.000000  30000.000000   
mean       5200.366300     16.097333      49.942467     37.066633   
std       16838.019547     75.076958     152.101430    107.0691

# 5. Export Queue for the Paper

The ranked recommendation queue is exported for reporting purposes.

The exported file includes:

- content identifier
- client identifier
- prediction score
- reason code

This file can be reused in the research paper and for operational review.

In [8]:
export_queue = queue[
    [
        "content_id",
        "client_id",
        "score",
        "reason_code"
    ]
]

export_queue.head()

,content_id,client_id,score,reason_code
0,content_5feee3994adb,client_7f2253d7e2,100,High predicted value
1,content_70450b1c27ae,client_3fdba35f04,90,High predicted value
2,content_5096a9d25fe5,client_6208ef0f77,90,High predicted value
3,content_d20b4e742dc6,client_19581e27de,90,High predicted value
4,content_c7fbadd2bf54,client_19581e27de,90,High predicted value


In [9]:
export_queue.to_csv(
    "content_action_queue.csv",
    index=False
)

print("CSV exported successfully.")

CSV exported successfully.


In [10]:
export_queue.head(15)

,content_id,client_id,score,reason_code
0,content_5feee3994adb,client_7f2253d7e2,100,High predicted value
1,content_70450b1c27ae,client_3fdba35f04,90,High predicted value
2,content_5096a9d25fe5,client_6208ef0f77,90,High predicted value
3,content_d20b4e742dc6,client_19581e27de,90,High predicted value
4,content_c7fbadd2bf54,client_19581e27de,90,High predicted value
5,content_5a67177a8e65,client_6208ef0f77,90,High predicted value
6,content_37c34ce699c3,client_19581e27de,90,High predicted value
7,content_88d367c507a3,client_3fdba35f04,90,High predicted value
8,content_dec180daa24f,client_6208ef0f77,90,High predicted value
9,content_4f81a5e449e2,client_6208ef0f77,90,High predicted value


# Self Check

✓ Ranked content queue created

✓ Reason codes generated

✓ Intended use documented

✓ Model limitations explained

✓ Human review checklist included

✓ No-go cases documented

✓ Monitoring and retraining triggers defined

✓ Export queue generated successfully